In [17]:
import numpy as np
import pandas as pd

import sys
import os

sys.path.append(os.path.abspath("../Generator"))
sys.path.append(os.path.abspath("../Reactions"))

import generator
import reactions


# Counters helper
def update_disease_counters(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    df.loc[df["dynamic.sirvStatus"] == "I", "dynamic.infectedDays"] += 1
    df.loc[df["dynamic.sirvStatus"] == "R", "dynamic.recoveredDays"] += 1
    df.loc[df["dynamic.sirvStatus"] == "V", "dynamic.vaccinatedDays"] += 1

    return df


# SIMULATION
def run_sirv_simulation(n_runs: int, m_steps: int):

    config = generator.read_config("../Config/config_sample.yml")
    df_init = generator.intialization()

    all_runs = []

    for run_id in range(n_runs):

        print(f"\n=== RUN {run_id + 1}/{n_runs} ===")

        rng = np.random.default_rng(config["seed"] + run_id)
        df = df_init.copy()
        run_history = []

        t0 = df.copy()
        t0["run"] = run_id
        t0["t"] = 0
        run_history.append(t0)       

        for t in range(m_steps):
            # movement
            df = generator.jiggle_positions(
                df,
                config["rho"],
                config["sig2"],
                rng
            )

            # disease dynamic
            df = reactions.susceptible_to_infected(df, config["sig2"])
            df = reactions.infected_to_recovered(df, config["rcd"])
            df = reactions.recovered_to_susceptible(df, config["sd"])
            df = reactions.vaccinated_to_susceptible(df, config["ved"])
            df = reactions.susceptible_to_vaccinated(df, target_fraction=0.57, n_days=180, rng=rng)

            # update counters
            df = update_disease_counters(df)

            snapshot = df.copy()
            snapshot["run"] = run_id
            snapshot["t"] = t + 1
            run_history.append(snapshot)

        run_df = pd.concat(run_history, ignore_index=True)
        all_runs.append(run_df)

    all_runs_df = pd.concat(all_runs, ignore_index=True)

    static_cols = [c for c in all_runs_df.columns if c.startswith("static.")]
    dynamic_numeric_cols = [c for c in all_runs_df.select_dtypes(include="number").columns
                            if c not in ["run", "t"] and not c.startswith("static.")]

    bool_cols = [c for c in all_runs_df.select_dtypes(include="boolean").columns
                 if not c.startswith("static.")]

    numeric_avg = all_runs_df.groupby(static_cols + ["t"])[dynamic_numeric_cols + bool_cols].mean().reset_index()
    
    sirv_avg = (
        all_runs_df.groupby(static_cols + ["t", "dynamic.sirvStatus"])
        .size()
        .unstack(fill_value=0)
        / n_runs
    ).reset_index()

    averaged = numeric_avg.merge(sirv_avg, on=static_cols + ["t"])

    return averaged

df_stats = run_sirv_simulation(n_runs=5, m_steps=100)

df_stats.to_csv("sirv_summary.csv", index=False)

Configuration and schema loaded successfully.
Generated 1000 synthetic population records.
Population size: 1000
Infected initial population: 10 individuals.
Synthetic population generated successfully.
Initial positions jiggle applied successfully.

=== RUN 1/5 ===

=== RUN 2/5 ===

=== RUN 3/5 ===

=== RUN 4/5 ===

=== RUN 5/5 ===


In [18]:
# Remove restrictions on maximum rows and columns
pd.set_option('display.max_rows', 10)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

# Display your dataframe
display(df_stats)

,static.guid,static.age,static.ageRiskMultiplier,static.comorbidity,static.comorbidityRiskMultiplier,static.socialActivity,static.socialActivityRiskMultiplier,static.geography,static.geographyRiskMultiplier,static.mobility,static.mobilityRiskMultiplier,static.vaccineAcceptance,static.vaccineAcceptanceRiskMultiplier,t,dynamic.numberOfInfections,dynamic.infectedDays,dynamic.vaccinatedDays,dynamic.recoveredDays,dynamic.currentLocation.xcor,dynamic.currentLocation.ycor,dynamic.vaccineStatus,dynamic.proactiveVaccine,I,R,S,V
0,000c380b-0650-4241-9139-20929650c0e5,0-17,0.50,True,1.0,low,0.5,urban,1.0,independent,1.0,True,0.5,0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,1.0,0.0
1,000c380b-0650-4241-9139-20929650c0e5,0-17,0.50,True,1.0,low,0.5,urban,1.0,independent,1.0,True,0.5,1,0.0,0.0,1.0,0.0,9.274176,18.225101,0.0,0.0,0.0,0.0,0.0,1.0
2,000c380b-0650-4241-9139-20929650c0e5,0-17,0.50,True,1.0,low,0.5,urban,1.0,independent,1.0,True,0.5,2,0.0,0.0,2.0,0.0,27.075507,9.344171,0.0,0.0,0.0,0.0,0.0,1.0
3,000c380b-0650-4241-9139-20929650c0e5,0-17,0.50,True,1.0,low,0.5,urban,1.0,independent,1.0,True,0.5,3,0.0,0.0,3.0,0.0,18.060194,17.846556,0.0,0.0,0.0,0.0,0.0,1.0
4,000c380b-0650-4241-9139-20929650c0e5,0-17,0.50,True,1.0,low,0.5,urban,1.0,independent,1.0,True,0.5,4,0.0,0.0,4.0,0.0,26.650404,35.356167,0.0,0.0,0.0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
100995,ffb6a1b9-17d4-461d-ace8-2334926d87e9,50-64,0.75,False,0.5,low,0.5,urban,1.0,independent,1.0,True,0.5,96,0.0,0.0,3.8,0.0,24.258100,28.190320,0.0,0.0,0.0,0.0,0.0,1.0
100996,ffb6a1b9-17d4-461d-ace8-2334926d87e9,50-64,0.75,False,0.5,low,0.5,urban,1.0,independent,1.0,True,0.5,97,0.0,0.0,4.8,0.0,24.645696,27.962812,0.0,0.0,0.0,0.0,0.0,1.0
100997,ffb6a1b9-17d4-461d-ace8-2334926d87e9,50-64,0.75,False,0.5,low,0.5,urban,1.0,independent,1.0,True,0.5,98,0.0,0.0,5.8,0.0,25.062694,28.067336,0.0,0.0,0.0,0.0,0.0,1.0
100998,ffb6a1b9-17d4-461d-ace8-2334926d87e9,50-64,0.75,False,0.5,low,0.5,urban,1.0,independent,1.0,True,0.5,99,0.0,0.0,6.8,0.0,24.248397,36.261989,0.0,0.0,0.0,0.0,0.0,1.0


In [19]:
print(df_stats.shape)

(101000, 26)
